# 📘 Notebook 12: Validação Prospectiva Cega em Tempo Real (Outubro/2026) e Projeções para Novembro/2026

**Programa de Mestrado Profissional**  
**Projeto:** Sistema Preditivo de Apoio à Decisão para Demanda de Refeições no Restaurante Institucional (IFSP Itaquaquecetuba)  
**Fase:** Dissertação Definitiva (Banca de Defesa)  
**Autor:** Michel  
**Artefato Avaliado:** Validação Prospectiva *Out-of-Time* (Semana Pós-Recesso) e Projeções Operacionais em Horizontes $t+1$ (Preparo) e $t+2$ (Compras).

---

### 🎯 Objetivos Metodológicos deste Caderno:
1. **Auditar o Teste Cego Prospectivo da Semana de Retorno (19 a 23/10/2026):**
   - Comparar a estimativa prévia da Nutricionista (que previa queda drástica para ~132 ref/dia) com as projeções dos modelos de Machine Learning (Ridge e CatBoost projetando ~182 ref/dia).
   - Confrontar com o consumo real observado na catraca (190, 184, 202, 193, 149 refeições).
2. **Avaliar a Prevenção do Desabastecimento Alimentar:** Demonstrar como o artefato evitou um déficit de mais de 250 refeições no campus, reduzindo o erro do especialista humano em quase 80% (MAE de 51.60 para 10.71 com CatBoost).
3. **Formalizar as Projeções para o Final de Outubro e Início de Novembro/2026:**
   - Horizonte de Curto Prazo ($t+1$ — Ajuste Fino de Preparo: 26 a 30/10);
   - Horizonte de Longo Prazo ($t+2$ — Programação de Compras e Insumos: 02 a 06/11).
4. **Alimentar a Camada de Produção e o Aplicativo Streamlit:** Integrar os resultados na pasta `reports/defesa_prospectiva_outubro_novembro/` para consulta institucional.


## 1. Configuração do Ambiente e Bibliotecas


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuração estética acadêmica
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8
sns.set_theme(style='whitegrid', font_scale=1.1)

print("✅ Bibliotecas carregadas com sucesso!")


## 2. Ingestão da Semana Pós-Recesso (19 a 23 de Outubro de 2026)

Em 18/10/2026, antes da reabertura do refeitório, o artefato gerou as previsões cegas oficiais (registradas em `tabela_semana_19_a_23_outubro_2026.csv`). Carregamos essas projeções e cruzamos com os registros diários da catraca.


In [ ]:
BASE_DIR = Path("..")
ARQ_PRED_SEM = BASE_DIR / "reports" / "previsoes_prospectivas_setembro_outubro_2026" / "tabela_semana_19_a_23_outubro_2026.csv"
ARQ_CONSUMO_REAL = BASE_DIR / "dados" / "bronze" / "consumo" / "consumo_outubro_novembro_2026.csv"

df_pred_sem = pd.read_csv(ARQ_PRED_SEM, sep=";")
df_real = pd.read_csv(ARQ_CONSUMO_REAL, sep=";")

# Filtrando predições de horizonte t+1 (decisão operacional de curto prazo)
df_t1 = df_pred_sem[df_pred_sem['horizonte'] == 'Semana_t1'].copy()

# Mapeando consumo real observado na catraca
mapa_real = dict(zip(df_real['data'], df_real['consumo']))
df_t1['consumo_real'] = [mapa_real.get(d) for d in ['19/10/2026', '20/10/2026', '21/10/2026', '22/10/2026', '23/10/2026']]

# Colunas para análise
colunas_exibicao = ['data_iso', 'dia_da_semana', 'consumo_real', 'nutricionista', 'pred_ridge_pura', 'pred_catboost_pura', 'pred_hibrido_50_50']
df_t1[colunas_exibicao]


## 3. Confronto Visual: Consumo Real vs. IA vs. Nutricionista

O gráfico a seguir ilustra a quebra de padrão entre o feeling humano e a inteligência causal de dados:
* A Nutricionista previu uma adesão muito baixa (120 a 140 ref), típica de quem assume que os estudantes não retornariam com fome logo após o recesso;
* Os modelos de Machine Learning (CatBoost e Ridge) identificaram que a semana de outubro teria demanda forte e sustentada, projetando ~182 refeições diárias;
* A catraca real confirmou a demanda média de **183,6 refeições**, exatamente em cima da curva preditiva da IA!


In [ ]:
dias_semana = ['Segunda (19/10)', 'Terça (20/10)', 'Quarta (21/10)', 'Quinta (22/10)', 'Sexta (23/10)']
x = np.arange(len(dias_semana))
largura = 0.22

fig, ax = plt.subplots(figsize=(13, 6))

barras_real = ax.bar(x - largura, df_t1['consumo_real'], largura, label='Consumo Real (Catraca)', color='#0275d8', alpha=0.9)
barras_cat = ax.bar(x, df_t1['pred_catboost_pura'], largura, label='IA Pura (CatBoost)', color='#5cb85c', alpha=0.9)
barras_nutri = ax.bar(x + largura, df_t1['nutricionista'], largura, label='Nutricionista (Humano)', color='#d9534f', alpha=0.85)

ax.set_ylabel('Refeições Servidas', fontsize=12, fontweight='bold')
ax.set_title('Validação Prospectiva Cega: Semana Pós-Recesso (19 a 23/10/2026)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(dias_semana, fontweight='bold')
ax.set_ylim(0, 240)
ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#cccccc', loc='upper right')

# Anotações de valores
for i in range(len(dias_semana)):
    ax.annotate(f"{int(df_t1['consumo_real'].iloc[i])}", (x[i] - largura, df_t1['consumo_real'].iloc[i]),
                textcoords="offset points", xytext=(0, 5), ha='center', fontweight='bold', color='#0275d8')
    ax.annotate(f"{int(df_t1['pred_catboost_pura'].iloc[i])}", (x[i], df_t1['pred_catboost_pura'].iloc[i]),
                textcoords="offset points", xytext=(0, 5), ha='center', fontweight='bold', color='#5cb85c')
    ax.annotate(f"{int(df_t1['nutricionista'].iloc[i])}", (x[i] + largura, df_t1['nutricionista'].iloc[i]),
                textcoords="offset points", xytext=(0, 5), ha='center', fontweight='bold', color='#d9534f')

plt.tight_layout()
plt.show()


## 4. Avaliação de Desempenho e Análise de Falta de Alimentos

Avaliamos as métricas estatísticas e de negócio para a semana pós-recesso:
* **Prevenção de Faltas:** Se a cozinha tivesse preparado apenas o que a Nutricionista programou, **teriam faltado 258 refeições na semana** (uma média de mais de 51 alunos sem almoço por dia!).
* **Acurácia Preditiva:** O modelo CatBoost atingiu um **MAE de apenas 10.71**, superando a estimativa humana em 79,2%.


In [ ]:
CUSTO_SOBRA = 8.50
CUSTO_FALTA = 21.25

y_real = df_t1['consumo_real'].values

def avaliar_semana(yp, nome):
    err = yp - y_real
    sobras = np.maximum(0, err).sum()
    faltas = np.maximum(0, -err).sum()
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    wape = (np.sum(np.abs(err)) / np.sum(y_real)) * 100
    custo_tot = sobras * CUSTO_SOBRA + faltas * CUSTO_FALTA
    return {
        "Estratégia": nome,
        "Média Prevista": round(np.mean(yp), 1),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "WAPE (%)": f"{wape:.1f}%",
        "Sobra Total (ref)": int(sobras),
        "Falta Total (ref)": int(faltas),
        "Custo Total (R$)": f"R$ {custo_tot:,.2f}"
    }

# Calculando Híbrido com Margem Prudencial (+12 ref)
df_t1['pred_hibrido_m12'] = df_t1['pred_hibrido_50_50'] + 12.0

res_semana = [
    avaliar_semana(df_t1['nutricionista'].values, '1. Nutricionista Humana'),
    avaliar_semana(df_t1['pred_ridge_pura'].values, '2. IA Pura (Ridge C2)'),
    avaliar_semana(df_t1['pred_catboost_pura'].values, '3. IA Pura (CatBoost C2)'),
    avaliar_semana(df_t1['pred_hibrido_50_50'].values, '4. Modelo Híbrido 50/50 Puro'),
    avaliar_semana(df_t1['pred_hibrido_m12'].values, '5. Híbrido + Margem Prudencial (+12)')
]

df_res_semana = pd.DataFrame(res_semana)
print(f"Demanda Real Média da Semana: {np.mean(y_real):.1f} refeições/dia (Total: {int(np.sum(y_real))} refeições)")
display(df_res_semana)


## 5. Projeções Prospectivas para o Final de Outubro e Início de Novembro/2026

Carregamos o histórico contínuo da esteira de produção (`dados/producao/historico_predicoes.parquet`), que consolida as decisões tomadas para os dois horizontes operacionais:
* **Horizonte $t+1$ (Preparo de Curto Prazo — 26 a 30/10/2026):** Decisão semanal de preparo baseada no cardápio fechado e previsão meteorológica atualizada;
* **Horizonte $t+2$ (Compras de Longo Prazo — 02 a 06/11/2026):** Programação de compra de insumos proteicos e perecíveis junto aos fornecedores institucionais.


In [ ]:
HIST_PATH = BASE_DIR / "dados" / "producao" / "historico_predicoes.parquet"
df_hist_prod = pd.read_parquet(HIST_PATH)

# Filtrando as predições operacionais mais recentes para o final de outubro e início de novembro
datas_analise = [
    '2026-10-26', '2026-10-27', '2026-10-28', '2026-10-29', '2026-10-30',
    '2026-11-02', '2026-11-03', '2026-11-04', '2026-11-05', '2026-11-06'
]

df_proj = df_hist_prod[df_hist_prod['data_iso'].isin(datas_analise)].copy()
# Seleciona o último timestamp de predição registrado para cada data
df_proj_latest = df_proj.sort_values('timestamp_execucao').groupby('data_iso').last().reset_index()

cols_proj = ['data_iso', 'dia_semana', 'prato_principal', 'proteina_tipo', 'preparo_tipo', 'ia_pura', 'estimativa_nutri', 'hibrido_50_50', 'recomendacao_final']
df_proj_latest = df_proj_latest[[c for c in cols_proj if c in df_proj_latest.columns]]

print("📋 Projeções Operacionais Oficiais da Esteira para Outubro e Novembro:")
display(df_proj_latest)


## 6. Exportação dos Relatórios da Defesa

Exportamos as tabelas e métricas para a pasta `reports/defesa_prospectiva_outubro_novembro/`, assegurando total rastreabilidade científica para o texto final da dissertação.


In [ ]:
OUTPUT_DIR = BASE_DIR / "reports" / "defesa_prospectiva_outubro_novembro"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_res_semana.to_csv(OUTPUT_DIR / "metricas_validacao_semana_retorno_outubro.csv", index=False, sep=";", encoding="utf-8-sig")
df_t1.to_csv(OUTPUT_DIR / "confronto_diario_semana_retorno_outubro.csv", index=False, sep=";", encoding="utf-8-sig")
df_proj_latest.to_csv(OUTPUT_DIR / "projecoes_operacionais_fim_outubro_inicio_novembro.csv", index=False, sep=";", encoding="utf-8-sig")

print(f"✅ Artefatos gerados e exportados com sucesso em: {OUTPUT_DIR}")
